# ReLoG — Federated Recommender

This notebook runs the **ReLoG** model with multiple random *seeds* to estimate
its average performance and variance.

ReLoG is a **federated** recommendation system with a *two-tower* architecture:

- a *shared* **User Tower** and **Item Tower** (aggregated across clients)
  that project text embeddings into a common space;
- a **local score function** (`client_mlp`), customized for each user and
  maintained on the individual client, which estimates user–item relevance.

Training follows the *Federated Averaging* paradigm (with momentum): in each
round, a subset of users trains its own model locally, and the weights
of the two towers are aggregated globally. Evaluation distinguishes between
**warm users** (seen during training) and **unseen users**, thereby measuring
*few-shot* adaptability.

In [1]:
# IMPORT

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import copy
import random
import math

In [ ]:
def set_seed(seed):
    """Set the seeds for Python, NumPy, and PyTorch to ensure reproducible runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# change path as you need
df_sampled      = pd.read_parquet('../../preprocessing/electronics_review.parquet') 
df_meta_aligned = pd.read_parquet('../../preprocessing/electronics_meta.parquet')

NUM_USERS    = df_sampled['user_id_int'].nunique()
NUM_ITEMS    = df_sampled['item_id_int'].nunique()
INTERACTIONS = len(df_sampled)
SPARSITY     = (1 - (INTERACTIONS / (NUM_USERS * NUM_ITEMS))) * 100

print(f"Users: {NUM_USERS}, Item: {NUM_ITEMS}, Interactions: {INTERACTIONS}")
print(f"Sparsity: {SPARSITY:.2f}%")
print(f"Columns df_sampled: {df_sampled.columns}")
print(f"Columns df_meta_aligned: {df_meta_aligned.columns}")
print(f"meta_text example: {df_meta_aligned.iloc[0]['meta_text']}")

## Calculating Embeddings

User reviews and item metadata records are encoded
using SBERT (`all-MiniLM-L6-v2`) into dense 384-dimensional vectors. These
embeddings serve as the input for the two towers: the reviews describe the user, and the
metadata describes the item.

In [ ]:
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=device)

df_sampled['summary']    = df_sampled['summary'].fillna('')
df_sampled['reviewText'] = df_sampled['reviewText'].fillna('')
df_sampled['full_text']  = df_sampled['summary'] + ". " + df_sampled['reviewText']

print("Calculating embeddings review...")
review_embeddings = sbert.encode(
    df_sampled['full_text'].tolist(),
    batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
review_emb_map = {i: review_embeddings[i] for i in range(len(df_sampled))}

print("Calculating embeddings metadata item...")
meta_embeddings = sbert.encode(
    df_meta_aligned['meta_text'].tolist(),
    batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
item_meta_tensor = torch.tensor(meta_embeddings, dtype=torch.float32)
print(f"Item bank: {item_meta_tensor.shape}")

item_meta_tensor_gpu = item_meta_tensor.to(device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Calcolo embeddings review...


Batches:   0%|          | 0/769 [00:00<?, ?it/s]

Calcolo embeddings metadati item...


Batches:   0%|          | 0/436 [00:00<?, ?it/s]

Item bank: torch.Size([27845, 384])


## Architecture Definition

The architecture is *two-tower*:

- **`UserTower`** and **`ItemTower`** project the SBERT embeddings into a
  common, L2-normalized latent space.
- **`LocalScoreFunction`** (`client_mlp`) is a small *local* MLP, specific
  to each user: it concatenates the user and item representations and produces the
  relevance score.

The model is trained using **BPR loss**, which ranks positive items above
negative ones.

In [ ]:
class UserTower(nn.Module):
    """User Tower: projects the review embeddings into latent space (L2-normalized)."""

    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            # nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class ItemTower(nn.Module):
    """Item Tower: projects the metadata embeddings into the same space as the user (L2-normalized)."""

    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            # nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, text_emb):
        return F.normalize(self.net(text_emb), dim=-1)


class LocalScoreFunction(nn.Module):
    """Local score function (per-user): Estimates the score for a (user, item) pair."""

    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, user_emb, item_emb):
        if user_emb.shape[0] != item_emb.shape[0]:
            if user_emb.shape[0] == 1:
                user_emb = user_emb.expand(item_emb.shape[0], -1)
            else:
                raise RuntimeError(f"Shape mismatch: {user_emb.shape} vs {item_emb.shape}")
        x = torch.cat([user_emb, item_emb], dim=-1)
        return self.net(x)


class TwoTowerRecommender(nn.Module):
    """Complete model: the two shared towers plus the local score function."""

    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256, inference_temperature=0.07):
        super().__init__()
        self.item_tower = ItemTower(input_dim, hidden_dim, output_dim)
        self.user_tower = UserTower(input_dim, hidden_dim, output_dim)
        self.client_mlp = LocalScoreFunction(input_dim=output_dim * 2, hidden_dim=128)
        self.inference_temperature = inference_temperature

    def get_user_repr(self, review_embeddings):
        return self.user_tower(review_embeddings).mean(dim=0, keepdim=True)

    def get_item_repr(self, item_meta_embeddings):
        return self.item_tower(item_meta_embeddings)

    def training_score(self, user_repr, item_reprs):
        raw_scores = self.client_mlp(user_repr, item_reprs).squeeze(-1)
        return raw_scores / self.inference_temperature

    def score(self, user_repr, item_reprs):
        return self.training_score(user_repr, item_reprs)


def bpr_loss(pos_scores, neg_scores):
    """Bayesian Personalized Ranking loss across all pairs (positive, negative).

    pos_scores: scores for positive items, shape [N_pos]
    neg_scores: scores for negative items, shape [N_neg]
    """
    diff = pos_scores.unsqueeze(1) - neg_scores.unsqueeze(0)
    return -F.logsigmoid(diff).mean()

## Data Utilities

`get_client_data` constructs a dataset for a single user using a
time-based split: the oldest interactions for training, the second-to-last for
validation, and the most recent for testing.

In [ ]:
def get_client_data(user_id, df, emb_map, mode="train"):
    """Prepares a user's data with a train/val/test split.

    Interactions are sorted by timestamp: all but the last two
    make up the train set, the second-to-last is the validation set, and the last one
    is the test set. Returns None if the user has fewer than 3 interactions.
    """
    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')
    if len(user_df) < 3:
        return None, None
    indices = user_df.index.tolist()
    train_idx = indices[:-2]
    val_idx   = indices[-2]
    test_idx  = indices[-1]
    X_train = torch.tensor(
        np.array([emb_map[i] for i in train_idx]),
        dtype=torch.float32
    )
    train_item_ids = user_df.loc[train_idx, 'item_id_int'].tolist()
    if mode == "val":
        target_id = int(user_df.loc[val_idx, 'item_id_int'])
    elif mode == "test":
        target_id = int(user_df.loc[test_idx, 'item_id_int'])
    else:
        target_id = None
    return (X_train, train_item_ids), target_id

## Hard Negative Sampling

Instead of random negatives, the model selects the uninteracted items to which the
model assigns the highest score: more informative “hard” negatives, which
make the BPR loss more effective.

In [ ]:
def sample_hard_negatives(user_repr, pos_set, all_metas, local_model,
                          num_neg, num_candidates, device, num_total_items):
    """Select hard negatives for the current user.

    Sample a pool of non-positive items, evaluate them using the local model, and
    return the `num_neg` items with the highest scores (the hardest negatives).
    """
    all_ids = np.arange(num_total_items)
    pos_arr = np.array(list(pos_set), dtype=np.int64)
    mask = np.ones(num_total_items, dtype=bool)
    mask[pos_arr] = False
    eligible = all_ids[mask]

    if len(eligible) < num_neg:
        return eligible.tolist()

    n_cands = min(num_candidates, len(eligible))
    candidate_ids = np.random.choice(eligible, size=n_cands, replace=False)

    with torch.no_grad():
        cand_ids_tensor = torch.tensor(candidate_ids, device=device)
        cand_metas = all_metas[cand_ids_tensor]
        cand_reprs = local_model.get_item_repr(cand_metas)
        cand_scores = local_model.training_score(user_repr.detach(), cand_reprs)

    top_k = min(num_neg, len(candidate_ids))
    top_indices = torch.topk(cand_scores, top_k).indices.cpu().numpy()
    return candidate_ids[top_indices].tolist()

## Local Client Training

Each client (user) starts with the global weights, trains the two towers and its
own score function locally, and returns the tower weights to be aggregated. The
learning rate follows a linear warmup followed by cosine annealing.

In [ ]:
def get_lr(base_lr, current_step, warmup_steps, total_steps):
    """Linear LR warmup → cosine annealing."""
    if current_step < warmup_steps:
        return base_lr * (current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

def train_client(user_id, global_state_dict, X_train_reviews, train_item_ids,
                 all_metas_gpu,   # item bank già residente su GPU
                 device, client_states,
                 lr=0.001, epochs=5, num_neg=10, use_hard_negatives=True,
                 lr_warmup_steps=10, current_step=0, total_steps=100):
    """Local training of a client (user) for a federated round.

    It starts with the global weights, trains the user's two towers and score function
    using BPR loss and hard negative sampling, and returns the shared weights (the
    towers) to be aggregated at the global level.
    """

    local_model = TwoTowerRecommender().to(device)
    local_model.load_state_dict(global_state_dict, strict=False)

    user_local_data = client_states.get(user_id, None)
    if user_local_data is not None:
        local_model.client_mlp.load_state_dict(user_local_data)

    effective_lr = get_lr(lr, current_step, lr_warmup_steps, total_steps)
    optimizer = torch.optim.Adam(local_model.parameters(), lr=effective_lr)
    local_model.train()

    X_train = X_train_reviews.to(device)
    pos_set = set(train_item_ids)
    num_total_items = all_metas_gpu.shape[0]

    pos_tensor = torch.tensor(train_item_ids, device=device)
    pos_metas  = all_metas_gpu[pos_tensor]   # shape: [N_pos, 384]

    loss = None
    for _ in range(epochs):
        optimizer.zero_grad()
        user_repr = local_model.get_user_repr(X_train)

        pos_reprs = local_model.get_item_repr(pos_metas)

        if use_hard_negatives:
            neg_ids = sample_hard_negatives(
                user_repr, pos_set, all_metas_gpu, local_model,
                num_neg=len(train_item_ids) * num_neg,
                num_candidates=2000, device=device,
                num_total_items=num_total_items
            )
        else:
            all_ids = np.arange(num_total_items)
            mask = np.ones(num_total_items, dtype=bool)
            mask[np.array(list(pos_set))] = False
            neg_ids = np.random.choice(
                all_ids[mask],
                size=len(train_item_ids) * num_neg,
                replace=True
            ).tolist()

        neg_tensor = torch.tensor(neg_ids, device=device)
        neg_metas  = all_metas_gpu[neg_tensor]
        neg_reprs  = local_model.get_item_repr(neg_metas)

        pos_scores = local_model.training_score(user_repr, pos_reprs)
        neg_scores = local_model.training_score(user_repr, neg_reprs)

        loss = bpr_loss(pos_scores, neg_scores)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
        optimizer.step()

    client_states[user_id] = local_model.client_mlp.state_dict()

    shared_state = {k: v.cpu() for k, v in local_model.state_dict().items()
                    if 'client_mlp' not in k}

    return shared_state, loss.item(), len(train_item_ids)

## Federated Aggregation (FedAvg + momentum)

The weights of the selected clients' towers are averaged in proportion to
the number of interactions for each. A *momentum* term stabilizes
the update of the global model across successive rounds. The score function
remains local to each user and is not aggregated.

In [ ]:
def weighted_fedavg_momentum(global_model, local_weights_list, local_sizes,
                             momentum_buffer, beta=0.9):
    """Aggregate client weights using weighted federated averaging + momentum.

    Each client is weighted in proportion to the number of interactions; only
    shared towers are aggregated (the client_mlp remains local). Momentum stabilizes
    the update of the global model across successive rounds.
    """
    total_samples = sum(local_sizes)
    global_dict   = global_model.state_dict()
    keys_to_agg   = [k for k in global_dict if 'client_mlp' not in k]

    if momentum_buffer is None:
        momentum_buffer = {k: torch.zeros_like(global_dict[k]) for k in keys_to_agg}

    with torch.no_grad():
        for key in keys_to_agg:
            layer_avg = torch.zeros_like(global_dict[key])
            for i, w in enumerate(local_weights_list):
                layer_avg.add_(w[key].to(layer_avg.device),
                               alpha=local_sizes[i] / total_samples)

            delta = layer_avg - global_dict[key]
            momentum_buffer[key].mul_(beta).add_(delta, alpha=1 - beta)
            global_dict[key].add_(momentum_buffer[key])

    global_model.load_state_dict(global_dict, strict=False)
    return momentum_buffer

## Evaluation (warm users)

**HR@k** and **NDCG@k** metrics using the leave-one-out protocol: for each user,
a brief local fine-tuning is performed, and then we check whether the target item is among the
first *k* recommended items.

In [ ]:
# =============================================================================
# VALUTAZIONE — OTTIMIZZATA
# =============================================================================

def evaluate_top_k(global_model, eval_users, df, emb_map,
                   all_metas_gpu,   # item bank già su GPU
                   client_states, k=10, device='cuda', mode="test",
                   eval_fraction=1.0):
    """Evaluate HR@k and NDCG@k using the leave-one-out protocol for warm users.

    For each user, perform a brief fine-tuning of the local score function,
    then rank the target against all other items and check whether it falls
    within the top-k. `mode` selects the target (‘val’ or ‘test’).
    """

    if eval_fraction < 1.0:
        n_sample   = max(1, int(len(eval_users) * eval_fraction))
        eval_users = np.random.choice(eval_users, n_sample, replace=False)
    
    num_items    = all_metas_gpu.shape[0]
    global_state = global_model.state_dict()
    all_ids_set  = set(range(num_items))
    all_ids_arr  = np.arange(num_items)

    global_model.eval()
    with torch.no_grad():
        chunk = 4096
        all_item_embs = torch.cat([
            global_model.get_item_repr(all_metas_gpu[i:i + chunk])
            for i in range(0, num_items, chunk)
        ], dim=0)  # [N_items, output_dim]

    hits, ndcgs, count = 0, 0, 0

    use_amp = (device == 'cuda')

    for user_id in tqdm(eval_users, desc=f"Evaluating ({mode})"):
        train_data, target_id = get_client_data(user_id, df, emb_map, mode=mode)
        if train_data is None:
            continue
        X_train, train_ids = train_data

        local_model = TwoTowerRecommender().to(device)
        local_model.load_state_dict(global_state, strict=False)

        user_local_data = client_states.get(user_id, None)
        if user_local_data is not None:
            local_model.client_mlp.load_state_dict(user_local_data)

        lr_eval = 0.005

        # --- LOCAL FINETUNING ---
        local_model.train()
        optimizer = torch.optim.Adam(local_model.client_mlp.parameters(), lr=lr_eval)
        X_train_dev = X_train.to(device)

        batch_pos = train_ids if len(train_ids) < 32 else random.sample(train_ids, 32)
        pos_t = torch.tensor(batch_pos, device=device)

        train_ids_set = set(train_ids)
        eligible_neg  = all_ids_arr[~np.isin(all_ids_arr, list(train_ids_set))]

        with torch.amp.autocast(device_type=device, enabled=use_amp):
            for _ in range(3):
                optimizer.zero_grad()
                user_repr = local_model.get_user_repr(X_train_dev)
                pos_reprs = local_model.get_item_repr(all_metas_gpu[pos_t])

                neg_idx = np.random.choice(eligible_neg, size=len(batch_pos), replace=False)
                neg_t   = torch.tensor(neg_idx, device=device)
                neg_reprs = local_model.get_item_repr(all_metas_gpu[neg_t])

                loss = bpr_loss(
                    local_model.training_score(user_repr, pos_reprs),
                    local_model.training_score(user_repr, neg_reprs)
                )
                loss.backward()
                optimizer.step()

        # --- INFERENCE ---
        local_model.eval()
        with torch.no_grad():
            user_repr = local_model.get_user_repr(X_train_dev)

            neg_cands   = list(all_ids_set - train_ids_set - {target_id})
            neg_arr     = np.array(neg_cands)
            neg_embs    = all_item_embs[neg_arr]  # shape: [N_neg, output_dim]
            target_emb  = all_item_embs[target_id].unsqueeze(0)  # [1, output_dim]

            with torch.amp.autocast(device_type=device, enabled=use_amp):
                target_score = local_model.score(user_repr, target_emb).item()
                neg_scores   = local_model.score(user_repr, neg_embs)

            all_scores = torch.cat([
                torch.tensor([target_score], device=device),
                neg_scores
            ])
            top_k_idx = torch.topk(all_scores, k).indices.cpu().numpy()

            if 0 in top_k_idx:
                hits += 1
                rank = int(np.where(top_k_idx == 0)[0][0])
                ndcgs += 1.0 / np.log2(rank + 2)
            count += 1

    if count == 0:
        return 0.0, 0.0
    return hits / count, ndcgs / count

## Few-shot evaluation (unseen users)

For users not seen during training, we measure the model’s adaptability: we
use the first *k* interactions (1-, 2-, or 3-shot, or all in “full” mode)
to fine-tune only the score function, and we evaluate the prediction
of the next interaction.

In [ ]:

def get_fewshot_data(user_id, df, emb_map, num_shots):
    """
    Returns (X_shots, shot_item_ids, target_id) for an unseen user.
    X_shots: embeddings of the first num_shots reviews [num_shots, 384]
    shot_item_ids: corresponding item IDs
    target_id: item ID of the (num_shots+1)-th interaction
    """
    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')
 
    if num_shots is None:
        if len(user_df) < 2:
            return None, None, None
        indices       = user_df.index.tolist()
        shot_idx      = indices[:-2] 
        target_idx    = indices[-1]
    else:
        min_required = num_shots + 1
        if len(user_df) < min_required:
            return None, None, None
        indices    = user_df.index.tolist()
        shot_idx   = indices[:num_shots]
        target_idx = indices[num_shots]  
 
    X_shots = torch.tensor(
        np.array([emb_map[i] for i in shot_idx]),
        dtype=torch.float32
    )
    shot_item_ids = user_df.loc[shot_idx, 'item_id_int'].tolist()
    target_id     = int(user_df.loc[target_idx, 'item_id_int'])
 
    return X_shots, shot_item_ids, target_id
 
 
def evaluate_fewshot(global_model, unseen_users, df, emb_map,
                     all_metas_gpu, num_shots,
                     k=10, device='cuda', finetune_epochs=5, lr=0.01):
    """
    Evaluate unseen users using few-shot adaptation.
    num_shots: int (1, 2, 3, ...) or None for “full”
    """
    label = f"{num_shots}-shot" if num_shots is not None else "full"
 
    num_items    = all_metas_gpu.shape[0]
    global_state = global_model.state_dict()
    all_ids_set  = set(range(num_items))
    all_ids_arr  = np.arange(num_items)
    use_amp      = (device == 'cuda')
 
    global_model.eval()
    with torch.no_grad():
        chunk = 4096
        all_item_embs = torch.cat([
            global_model.get_item_repr(all_metas_gpu[i:i + chunk])
            for i in range(0, num_items, chunk)
        ], dim=0)
 
    hits, ndcgs, count = 0, 0, 0
 
    for user_id in tqdm(unseen_users, desc=f"Few-shot eval ({label})", leave=False):
        X_shots, shot_item_ids, target_id = get_fewshot_data(
            user_id, df, emb_map, num_shots
        )
        if X_shots is None:
            continue
 
        local_model = TwoTowerRecommender().to(device)
        local_model.load_state_dict(global_state, strict=False)
 
        X_shots_dev   = X_shots.to(device)
        shot_ids_set  = set(shot_item_ids)
        eligible_neg  = all_ids_arr[~np.isin(all_ids_arr, list(shot_ids_set))]
 
        local_model.train()
        optimizer = torch.optim.Adam(local_model.client_mlp.parameters(), lr=lr)
 
        if len(shot_item_ids) > 0 and len(eligible_neg) > 0:
            batch_pos = shot_item_ids if len(shot_item_ids) < 32 \
                        else random.sample(shot_item_ids, 32)
            pos_t = torch.tensor(batch_pos, device=device)
 
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                for _ in range(finetune_epochs):
                    optimizer.zero_grad()
                    user_repr = local_model.get_user_repr(X_shots_dev)
                    pos_reprs = local_model.get_item_repr(all_metas_gpu[pos_t])
                    n_neg     = min(len(batch_pos), len(eligible_neg))
                    neg_idx   = np.random.choice(eligible_neg, size=n_neg, replace=False)
                    neg_t     = torch.tensor(neg_idx, device=device)
                    neg_reprs = local_model.get_item_repr(all_metas_gpu[neg_t])
                    loss = bpr_loss(
                        local_model.training_score(user_repr, pos_reprs),
                        local_model.training_score(user_repr, neg_reprs)
                    )
                    loss.backward()
                    optimizer.step()
 
        local_model.eval()
        with torch.no_grad():
            user_repr  = local_model.get_user_repr(X_shots_dev)
            neg_cands  = list(all_ids_set - shot_ids_set - {target_id})
            neg_arr    = np.array(neg_cands)
            neg_embs   = all_item_embs[neg_arr]
            target_emb = all_item_embs[target_id].unsqueeze(0)
 
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                target_score = local_model.score(user_repr, target_emb).item()
                neg_scores   = local_model.score(user_repr, neg_embs)
 
            all_scores = torch.cat([torch.tensor([target_score], device=device), neg_scores])
            top_k_idx  = torch.topk(all_scores, k).indices.cpu().numpy()
 
            if 0 in top_k_idx:
                hits += 1
                rank = int(np.where(top_k_idx == 0)[0][0])
                ndcgs += 1.0 / np.log2(rank + 2)
            count += 1
 
    if count == 0:
        return 0.0, 0.0
    print(f"  [{label}] evaluated users: {count}/{len(unseen_users)}")
    return hits / count, ndcgs / count

## User Split

Users are divided into **warm** (seen during training) and **unseen** (used
only for few-shot evaluation), in a way that is reproducible given the same seed.

In [ ]:
def split_users(df, unseen_ratio=0.2, seed=42):
    """Divides users into “train” (warm) and “unseen” groups, shuffling them in a reproducible manner."""
    rng = np.random.RandomState(seed)
    all_users = df['user_id_int'].unique()
    rng.shuffle(all_users)
    n_total = len(all_users)
    n_unseen  = int(n_total * unseen_ratio)
    unseen_users  = all_users[:n_unseen]
    train_users = all_users[n_unseen:]
    return train_users, unseen_users

## Experiment Definition

`run_experiment` orchestrates a complete run for a given seed: user splitting,
federated training for a fixed number of rounds with periodic evaluation,
selection of the best model on the validation set, and final testing on warm and unseen
users.

In [ ]:
def run_experiment(seed):
    """Runs an entire experiment (federated training + evaluation) for a seed.

    It returns metrics on warm users and few-shot results (1/2/3-shot and full)
    for unseen users, calculated using the best model selected during validation.
    """
    print(f"\n===== RUN with seed {seed} =====")
    set_seed(seed)

    train_users, unseen_users = split_users(
        df_sampled, unseen_ratio=0.2, seed=seed
    )
    print(f"Train users: {len(train_users)}")
    print(f"Unseen users:   {len(unseen_users)}")

    # -- CONFIGURATION --
    LR                    = 0.0005
    LOCAL_EPOCHS          = 3
    NUM_NEG_TRAIN         = 10
    USE_HARD_NEG          = True
    CLIENTS_PER_ROUND     = round(0.05 * len(train_users)) 
    GLOBAL_ROUNDS         = 100
    EVAL_EVERY            = 5
    INFERENCE_TEMPERATURE = 0.07
    FEDAVG_MOMENTUM       = 0.9
    K                     = 20
    EVAL_FRACTION         = 1
    LR_WARMUP_STEPS       = 10

    client_states   = {user_id: None for user_id in train_users}
    best_val_hr    = 0.0
    best_val_ndcg  = 0.0
    best_state      = None
    best_client_states = None
    momentum_buffer = None

    global_model = TwoTowerRecommender(
        inference_temperature=INFERENCE_TEMPERATURE
    ).to(device)

    print(f"\n=== Starting Federated training with seed = {seed} ===")
    print(f"{'Round':<6} | {'Loss':<8} | {'HR@' + str(K):<8} | {'NDCG@' + str(K):<8}")
    print("-" * 45)

    for round_num in range(1, GLOBAL_ROUNDS + 1):
        local_weights = []
        local_sizes   = []
        local_losses  = []

        round_state_dict = global_model.state_dict()

        selected = np.random.choice(train_users, CLIENTS_PER_ROUND, replace=False)
        for user_id in selected:
            train_data, _ = get_client_data(user_id, df_sampled, review_emb_map)
            if train_data is None:
                continue
            X_train, train_item_ids = train_data

            w, loss, n = train_client(
                user_id,
                round_state_dict,       
                X_train,
                train_item_ids,
                item_meta_tensor_gpu,   
                device,
                client_states,
                lr=LR,
                epochs=LOCAL_EPOCHS,
                num_neg=NUM_NEG_TRAIN,
                use_hard_negatives=USE_HARD_NEG,
                lr_warmup_steps=LR_WARMUP_STEPS,
                current_step=round_num - 1,
                total_steps=GLOBAL_ROUNDS
            )

            local_weights.append(w)
            local_sizes.append(n)
            local_losses.append(loss)

        if not local_weights:
            continue

        momentum_buffer = weighted_fedavg_momentum(
            global_model, local_weights, local_sizes,
            momentum_buffer, beta=FEDAVG_MOMENTUM
        )

        avg_loss = sum(local_losses) / len(local_losses)

        if round_num % EVAL_EVERY == 0:
            val_hr, val_ndcg = evaluate_top_k(
                global_model, train_users, df_sampled,
                review_emb_map, item_meta_tensor_gpu,   # item bank su GPU
                client_states, k=K, device=device, mode="val",
                eval_fraction=EVAL_FRACTION
            )

            marker = ""
            if val_hr > best_val_hr:
                best_val_hr        = val_hr
                best_state         = copy.deepcopy(global_model.state_dict())
                best_client_states = copy.deepcopy(client_states)
                marker = "  <- Best"
 
            print(f"{round_num:<6} | {avg_loss:<8.4f} | {val_hr:<10.4f} | {val_ndcg:<10.4f} {marker}")
        else:
            print(f"{round_num:<6} | {avg_loss:<8.4f} |")

    print("\n=== End Training ===")

    # =========================================================================
    # Final Test on best model
    # =========================================================================
    global_model.load_state_dict(best_state)
     
    print("\n--- TEST WARM USERS (ultima interazione) ---")
    warm_hr, warm_ndcg = evaluate_top_k(
        global_model, train_users, df_sampled,
        review_emb_map, item_meta_tensor_gpu,
        best_client_states, k=K, device=device, mode="test"
    )
    print(f"Warm  HR@{K}: {warm_hr:.4f}  |  NDCG@{K}: {warm_ndcg:.4f}")

    print("\n--- TEST UNSEEN USERS (few-shot adaptation) ---")
    shot_configs = [1, 2, 3, None]   # None = full
    fewshot_results = {}
 
    for num_shots in shot_configs:
        label = f"{num_shots}-shot" if num_shots is not None else "full"
        hr, ndcg = evaluate_fewshot(
            global_model, unseen_users, df_sampled,
            review_emb_map, item_meta_tensor_gpu,
            num_shots=num_shots, k=K, device=device,
            finetune_epochs=5, lr=0.005
        )
        fewshot_results[label] = (hr, ndcg)
        print(f"  {label:<8}  HR@{K}: {hr:.4f}  |  NDCG@{K}: {ndcg:.4f}")
 
    return warm_hr, warm_ndcg, fewshot_results

## Multi-seed Execution and Aggregated Results

The experiment is repeated using multiple seeds; the results are aggregated as
**mean ± standard deviation** for each scenario (warm and few-shot).

In [ ]:
seeds      = [0] # 0, 1, 2, 3, 4
shot_labels = ["1-shot", "2-shot", "3-shot", "full"]
 
warm_hrs, warm_ndcgs = [], []
fewshot_hrs  = {l: [] for l in shot_labels}
fewshot_ndcgs = {l: [] for l in shot_labels}
 
for s in seeds:
    warm_hr, warm_ndcg, fewshot_results = run_experiment(s)
    warm_hrs.append(warm_hr)
    warm_ndcgs.append(warm_ndcg)
    for label in shot_labels:
        fewshot_hrs[label].append(fewshot_results[label][0])
        fewshot_ndcgs[label].append(fewshot_results[label][1])
 
K = 20
 
print("\n" + "=" * 50)
print("RISULTATI FINALI (media ± std su 5 seed)")
print("=" * 50)
 
print(f"\n{'Scenario':<12} | {'HR@'+str(K):<18} | {'NDCG@'+str(K):<18}")
print("-" * 55)
 
# Warm users
m_hr   = np.mean(warm_hrs);   s_hr   = np.std(warm_hrs)
m_ndcg = np.mean(warm_ndcgs); s_ndcg = np.std(warm_ndcgs)
print(f"{'warm':<12} | {m_hr:.4f} ± {s_hr:.4f}   | {m_ndcg:.4f} ± {s_ndcg:.4f}")
 
# Few-shot unseen users
for label in shot_labels:
    m_hr   = np.mean(fewshot_hrs[label]);   s_hr   = np.std(fewshot_hrs[label])
    m_ndcg = np.mean(fewshot_ndcgs[label]); s_ndcg = np.std(fewshot_ndcgs[label])
    print(f"{label:<12} | {m_hr:.4f} ± {s_hr:.4f}   | {m_ndcg:.4f} ± {s_ndcg:.4f}")


===== RUN con seed 0 =====
Train users: 4352
Unseen users:   1087

=== Inizio Training Federato con seed = 0 ===
Round  | Loss     | HR@20    | NDCG@20 
---------------------------------------------
1      | 0.7070   |
2      | 0.6877   |
3      | 0.6754   |
4      | 0.6654   |


Evaluating (val): 100%|██████████| 4352/4352 [02:18<00:00, 31.31it/s]


5      | 0.6547   | 0.0193     | 0.0096       <- Best
6      | 0.6451   |
7      | 0.6362   |
8      | 0.6305   |
9      | 0.6220   |


Evaluating (val): 100%|██████████| 4352/4352 [02:17<00:00, 31.62it/s]


10     | 0.6117   | 0.0207     | 0.0108       <- Best
11     | 0.6071   |
12     | 0.5991   |
13     | 0.6013   |
14     | 0.5926   |


...

Warm  HR@20: 0.0149  |  NDCG@20: 0.0069

--- TEST UNSEEN USERS (few-shot adaptation) ---


  [1-shot] utenti valutati: 1087/1087
  1-shot    HR@20: 0.0460  |  NDCG@20: 0.0316


  [2-shot] utenti valutati: 1087/1087
  2-shot    HR@20: 0.0442  |  NDCG@20: 0.0259


  [3-shot] utenti valutati: 1087/1087
  3-shot    HR@20: 0.0460  |  NDCG@20: 0.0278


  [full] utenti valutati: 1087/1087
  full      HR@20: 0.0156  |  NDCG@20: 0.0090

RISULTATI FINALI (media ± std su 5 seed)

Scenario     | HR@20              | NDCG@20           
-------------------------------------------------------
warm         | 0.0149 ± 0.0000   | 0.0069 ± 0.0000
1-shot       | 0.0460 ± 0.0000   | 0.0316 ± 0.0000
2-shot       | 0.0442 ± 0.0000   | 0.0259 ± 0.0000
3-shot       | 0.0460 ± 0.0000   | 0.0278 ± 0.0000
full         | 0.0156 ± 0.0000   | 0.0090 ± 0.0000
